# M01C06: `sys.getsizeof` vs. True Footprint; `tracemalloc` Mastery

**System Information:** Apple M1 Max (macOS), Python 3.14.0

**Version Diagnostic:**

```python:disable-run
import sys
print(f"CPython {sys.version} ({sys.implementation.cache_tag})")
assert sys.implementation.name == 'cpython', "This module assumes CPython 3.12+; adjust for alternatives."
```

> **Guidance for Extension Developers:** This module focuses on CPython's measurement APIs. For portable extensions, avoid custom allocators; leverage `PyObject_SIZE` via stable C API (PEP 384) or HPy for object sizing. Consult [HPy documentation](https://hpyproject.org/) for cross-impl tracing.

This module demystifies memory footprint estimation, contrasting `sys.getsizeof`'s shallow view with `tracemalloc`'s comprehensive tracing. Building on M01C01–C05's model, we dissect allocator retention, recursive costs, and snapshot diffs, with implications for leak hunting and optimization. Explanations integrate empirical labs, historical rationale (e.g., `tracemalloc`'s 3.4 debut), and edge cases, ensuring self-contained rigor.

> **Fact Typing Convention:** Throughout this module, facts are classified as follows for clarity and longevity:
> - **Semantic Invariant:** Portable across Python implementations (e.g., PyPy); enshrined in the language specification.
> - **CPython Implementation Detail:** Specific to CPython 3.12–3.14; measured empirically; subject to evolution without notice.
> - **Forward-Looking Note:** Based on PEPs (e.g., 703); provisional until stabilized.

The content progresses from motivational gaps to tool mastery, cookbook recipes, and verification. All code is self-contained and executable in CPython 3.12+.

## Glossary of Key Terms

For quick reference, the following table defines abbreviations and concepts used throughout. Expand for full details.

<details>
<summary>Expand Glossary</summary>

| Term/Abbreviation | Definition |
|-------------------|------------|
| **`sys.getsizeof`** | Shallow size query: Reports object's logical allocation (header + payload), excluding refs/rounding. |
| **`tracemalloc`** | Heap tracer: Captures Python allocations with tracebacks; enables peak/snapshot diffs for attribution. |
| **Footprint** | True memory: Reachable objects + headers + rounding + arenas + non-Python (e.g., NumPy). |
| **Snapshot Diff** | Filtered comparison: Attributes net growth/frees to lines/modules; leak hunting essential. |
| **Arena Retention** | Pymalloc behavior: RSS high-water adherence; del/GC reuses, not OS-returns. |
| **nframe** | Traceback depth: `tracemalloc.start(nframe=10)`; balances detail vs. overhead. |
| **Filter** | Snapshot predicate: Includes/excludes by filename/path; isolates module costs. |
| **RSS** | Resident Set Size: OS-level truth; includes C/arenas beyond Python trace. |

</details>

## 0. Why You Care: Motivational Experiment

`sys.getsizeof` veils recursive/allocator costs; `tracemalloc` unveils peaks, but RSS reveals retention. This lab—deep tree build—exposes the chasm, motivating mastery below.

**Scope:** **CPython 3.12+** 64-bit releases; PyPy differs (JIT elides). Measurements empirical; qualitatives stable.

**Caveats:**
- Portable: Semantics (e.g., `__sizeof__`) standard; traces CPython-only.
- Production: `tracemalloc` ~10–50% overhead; sample for large runs.

**Version Matrix:** *(CPython Implementation Detail; Measured 3.12–3.14)*

| Version | tracemalloc Default nframe | Key Notes |
|---------|----------------------------|-----------|
| 3.12   | 1                         | Baseline tracing. |
| 3.13   | 1                         | Free-threaded hooks experimental. |
| 3.14   | 1                         | Stabilized; no shifts. |

**Guiding Questions:**
- Why `getsizeof` ~hundreds vs. peak ~MBs?
- Post-del/GC: Why tracemalloc drops, RSS lags?
- How to attribute to your code?

In [1]:
import tracemalloc
import sys
import gc

def build_tree(depth: int):
    if depth == 0:
        return object()  # Leaf
    return [build_tree(depth-1), build_tree(depth-1)]  # Full binary; no sharing

tracemalloc.start(5)  # Positional nframe for compatibility
root = build_tree(18)  # ~524K nodes; adjust to 15–17 if RecursionError
current, peak_during = tracemalloc.get_traced_memory()
print(f"Peak during construction (tracemalloc): {peak_during / 1e6:.1f} MB")
print(f"sys.getsizeof(root): {sys.getsizeof(root)} bytes")  # Top-level only

del root
gc.collect()
current_after, _ = tracemalloc.get_traced_memory()
print(f"tracemalloc current after del+gc: {current_after / 1e6:.1f} MB")
tracemalloc.stop()

Peak during construction (tracemalloc): 23.1 MB
sys.getsizeof(root): 72 bytes
tracemalloc current after del+gc: 0.0 MB


## 1. Scope Reminder

**CPython 3.12+** reality; other impls (PyPy) diverge. Builds on C01–C05: Headers/arenas under-report; now quantify.

## 2. What `sys.getsizeof(obj)` Actually Returns

**CPython Implementation Detail:** Delegates to `obj.__sizeof__()`: C-level for builtins (header + var + over-alloc); convention, omits secondary buffers.

User classes: Arbitrary; can lie.

**Lab 1: getsizeof Lies Everywhere**

Empty list; big with distinct objects: Shallow ignores nests. *(Semantic Invariant: `__sizeof__` convention; portable.)*

In [2]:
import sys

class Liar:
    def __sizeof__(self): return 0

print(sys.getsizeof(Liar()))  # 0; but consumes

empty_list = []
print(sys.getsizeof(empty_list))  # 56

big_list = [object() for _ in range(1_000_000)]  # Distinct!
print(sys.getsizeof(big_list))  # ~8–9 MB (ptrs only)

# Expected: Shallow; elements extra MBs ignored.

32
56
8448728


## 3. When `sys.getsizeof` Is Still Useful

Homogeneous containers (no nests): Tuples exact; lists include over-alloc (~1.125×). Cached small ints: Zero extra.

| Type | Counts | Example |
|------|--------|---------|
| tuple(range(1000)) | Exact | No over |
| list(range(1000)) | +Over | ~1125% |
| bytes(1000) | Exact | Compact |

**Pitfall:** Everywhere else, top-level only—rarely sufficient.

## 4. `tracemalloc`: Primary Tool for Python-Level Allocations

**CPython Implementation Detail:** Hooks `PyMem_*`/`PyObject_*`; records with tracebacks. Raw current/peak process-wide since start; snapshots attribute via diff/filter.

**Overhead:** ~10–50% CPU; own memory. Tip: `nframe=10` (default 1 shallow).

**Blinds:** Pre-start; non-Python (NumPy `nbytes`, mmap, C heaps, GPU).

**Rule:** RSS >> tracemalloc → Non-Python dominates.

**Patterns:**
1. Peak: `start(); fn(); peak`.
2. Diff: `snap1 = take_snapshot(); fn(); snap2.compare_to(snap1, 'lineno')[:10]`.

**Lab 2: Precise DS Measurement**

Slotted vs. dict tree: Slots < dicts (40–60%). *(CPython Implementation Detail: Dict over.)*

In [3]:
import tracemalloc

def measure_peak(fn):
    tracemalloc.start()
    fn()
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return peak

class SNode:
    __slots__ = ('left', 'right', 'value')
    def __init__(self, v): self.value = v; self.left = self.right = None

class DNode:
    def __init__(self, v): self.value = v; self.left = self.right = None

def build(NodeCls, n):
    nodes = [NodeCls(i) for i in range(n)]
    for i in range(n-1):
        nodes[i].left = nodes[i+1]
    return nodes[0]

s_peak = measure_peak(lambda: build(SNode, 1_000_000))
d_peak = measure_peak(lambda: build(DNode, 1_000_000))

print(f"Slotted peak: {s_peak/1e6:.1f} MB")
print(f"Dict    peak: {d_peak/1e6:.1f} MB")  # 2–3× higher

# Expected: Slots cheaper.

Slotted peak: 96.4 MB
Dict    peak: 136.4 MB


**Lab 3: Hogs via Filtering + Diff**

Snapshot pre/post-build; filter module; top net growth. *(CPython Implementation Detail: lineno attrib.)*

In [4]:
import tracemalloc

# Definitions from Lab 2 (for completeness)
class SNode:
    __slots__ = ('left', 'right', 'value')
    def __init__(self, v): self.value = v; self.left = self.right = None

def build(NodeCls, n):
    nodes = [NodeCls(i) for i in range(n)]
    for i in range(n-1):
        nodes[i].left = nodes[i+1]
    return nodes[0]

# Snapshot comparison
tracemalloc.start(10)  # Positional nframe for compatibility
snap1 = tracemalloc.take_snapshot()

obj = build(SNode, 1_000_000)  # Workload
snap2 = tracemalloc.take_snapshot()
tracemalloc.stop()

# Filter (adjust '*c06*' to your file)
filters = (
    tracemalloc.Filter(True, "*c06*"),
    tracemalloc.Filter(False, "<frozen importlib._bootstrap>"),
    tracemalloc.Filter(False, "tracemalloc.py"),
)

snap1 = snap1.filter_traces(filters)
snap2 = snap2.filter_traces(filters)

top = snap2.compare_to(snap1, 'lineno')[:10]
for stat in top:
    print(stat)  # Positive: Net growth (leaks)

## 5. Recursive Footprint (Poor Man's tracemalloc)

Python walk: `deep_sizeof`/`deep_sizeof_gc`; slow on large graphs; misses rounding/arenas. Lower bound. *(CPython Implementation Detail: `gc.get_referents` incomplete.)*

In [5]:
import sys
import gc

def deep_sizeof(obj, seen=None):
    if seen is None:
        seen = set()
    oid = id(obj)
    if oid in seen:
        return 0
    seen.add(oid)
    size = sys.getsizeof(obj)
    if isinstance(obj, (list, tuple, set, frozenset)):
        size += sum(deep_sizeof(i, seen) for i in obj)
    elif isinstance(obj, dict):
        size += sum(deep_sizeof(k, seen) + deep_sizeof(v, seen) for k, v in obj.items())
    elif hasattr(obj, '__dict__'):
        size += deep_sizeof(obj.__dict__, seen)
    elif hasattr(obj, '__slots__'):
        size += sum(deep_sizeof(getattr(obj, slot, None), seen) for slot in obj.__slots__)
    return size

def deep_sizeof_gc(obj, seen=None):
    if seen is None:
        seen = set()
    oid = id(obj)
    if oid in seen:
        return 0
    seen.add(oid)
    size = sys.getsizeof(obj)
    for ref in gc.get_referents(obj):
        size += deep_sizeof_gc(ref, seen)
    return size

# Sample tree definition (depth=5; ~63 nodes; adjust for scale)
def build_sample_tree(depth: int):
    if depth == 0:
        return object()  # Leaf
    return [build_sample_tree(depth - 1), build_sample_tree(depth - 1)]

root = build_sample_tree(5)  # Small tree for demonstration

# Example usage
print(f"deep_sizeof(root): {deep_sizeof(root):,} B")
print(f"deep_sizeof_gc(root): {deep_sizeof_gc(root):,} B")

# Caveat: Slow for large/unbounded graphs; use on known structures only.

deep_sizeof(root): 2,744 B
deep_sizeof_gc(root): 2,744 B


## 6. Cookbook: “How Big Is My DS?” (Every Time)

1. RSS baseline (e.g., `psutil`).
2. `tracemalloc.start(nframe=10)`; build; peak + filtered snap diff.
3. Optional: `deep_sizeof_gc(root)` ballpark.
4. Del + `gc.collect()`; final snap/RSS.
5. Interpret: Diff = attributable; peak = Python high-water; RSS delta = truth (arenas/C).

Prefer peak + filtered diff.

**Exercise (Mandatory):** Non-trivial DS (e.g., tree/list/graph) in three layouts: Dict/slotted/immutable. Cookbook full; report diffs via headers/rounding/`__dict__`/arenas. *(CPython Implementation Detail: Dict 2–3× slotted.)*

## 8. Verification: pytest Suite – Memory Invariants

Asserts: List ignores elements; slotted < dicts. *(CPython Implementation Detail: Modest N; order stable.)*

In [6]:
import sys
import tracemalloc
import pytest

def measure_peak(fn):
    tracemalloc.start()
    fn()
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return peak

def test_list_does_not_count_elements():
    lst = [object() for _ in range(100_000)]
    assert sys.getsizeof(lst) < 2_000_000  # Ptrs only

@pytest.mark.slow
def test_slotted_saves_memory():
    class S: 
        __slots__ = ('x',); 
        def __init__(self): 
            self.x = 1
    class D: 
        def __init__(self): 
            self.x = 1

    def build(Cls): 
        return [Cls() for _ in range(100_000)]

    s_peak = measure_peak(lambda: build(S))
    d_peak = measure_peak(lambda: build(D))
    assert s_peak < d_peak  # 40–60%

# Run (pytest style)
suite = [test_list_does_not_count_elements, test_slotted_saves_memory]
for test in suite:
    test()
print("\nSuite: Footprint invariants hold.")


Suite: Footprint invariants hold.


## 9. Essential Takeaways: Invariants to Internalize

1. **`getsizeof` Shallow (CPython Implementation Detail):** Payload only; no refs/rounding.
2. **True Footprint:** Objects + headers + rounding + arenas + non-Python.
3. **`tracemalloc` Canonical (CPython Implementation Detail):** Diff/filter for attrib; combine RSS.
4. **Peak + Diff Priority:** Current process-wide; attribution via lineno.
5. **Bugs = Alloc Bugs:** Profile targeted; RSS truth.
6. **Re-Measure (Semantic Invariant):** Version/platform vary; qualitatives stable.

**Synthesis:** Tools bridge model to reality; complete stack.

## 10. Progression: Subsequent Modules in M01

Footprint quantified; refine layouts:

- **M01C07: Dict vs. Slots vs. Tuples** – Per-instance costs.
- **M01C08+: Lookup, Descriptors** – Dispatch overheads.

Measurement discipline endures.